# Drain parser for log analysis

In [ ]:
import os, sys
from datetime import datetime
from pathlib import Path

sys.path.insert(0, os.path.abspath(".."))  # project root

# ── Run tag ───────────────────────────────────────────────────────────────────
# Auto-generated as YYYYMMDD_HHMM so multiple daily runs stay distinct.
# Override before running to reload or continue a specific experiment:
#   RUN_TAG = "20260407_1030"   # exact timestamp
#   RUN_TAG = "20260407_v3"     # manual version label
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M")

# Convenience: also expose individual parts if needed downstream
RUN_DATE = RUN_TAG.split("_")[0]

print(f"Run tag : {RUN_TAG}")
print(f"Outputs will be prefixed with: {RUN_TAG}_")


In [ ]:
from src.parser import DrainParser

config_path = '../configs/drain.ini'
hdfs_log_path = "../data/raw/HDFS_full.log"

n_lines = sum(1 for _ in open(hdfs_log_path, "rb"))
print(f"Total lines in log: {n_lines:,}")

parser = DrainParser(config_path=config_path)

# First pass: fit the parser to the log file, building the template clusters
print("Fitting parser to log file...")
parser.fit_file(hdfs_log_path)

# Second pass: annotate every line with its final cluster_id, template, params, block_id
print("Annotating log file...")
df = parser.annotate_file(hdfs_log_path, max_lines=None)
print(df.shape)
df.head(3)

parser.export_templates(f"../data/processed/HDFS/{RUN_TAG}_hdfs_templates.json")
parser.save(f"../models/HDFS/{RUN_TAG}_drain_parser.bin")


## Persistence

In [ ]:
import json
from pathlib import Path

PROCESSED_DIR = Path("../data/processed/HDFS")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_PATH  = PROCESSED_DIR / f"{RUN_TAG}_hdfs_annotated.parquet"

# Parquet cannot natively store Python lists in all engines,
# so encode the `parameters` column as a JSON string.
df_save = df.copy()
df_save["parameters"] = df_save["parameters"].apply(json.dumps)

# fastparquet cannot write pandas Arrow-backed string columns (ArrowDtype).
# The error "Unable to avoid copy while creating an array as requested" is
# caused by those columns. Cast all string/object columns to plain numpy object.
for col in df_save.select_dtypes(include=["object", "string"]).columns:
    df_save[col] = df_save[col].astype(object)

# Use fastparquet to avoid the pandas/pyarrow version-conflict bug
# (ArrowKeyError: arrow.py_extension_type) that occurs when pyarrow
# is loaded after pandas in the same kernel session.
df_save.to_parquet(PARQUET_PATH, index=False, engine="fastparquet")

print(f"Saved {len(df_save):,} rows  →  {PARQUET_PATH}")
print(f"File size : {PARQUET_PATH.stat().st_size / 1_048_576:.1f} MB")
print(f"Columns   : {df_save.columns.tolist()}")


In [ ]:
parser.validate()

## NEXT STEP: Enrich the mained templates with semantic information

Use LLM with sourced context to expand the template with semantic information for better embedding

See 2_EnrichTemplates.ipynb